In [ ]:
import pandas as pd
import numpy as np
import missingno as msno
import matplotlib.pyplot as plt
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import MissingIndicator, KNNImputer,SimpleImputer
from sklearn.impute import IterativeImputer
from sklearn.linear_model import BayesianRidge

import seaborn as sns

In [ ]:
# --- Portable paths (added for the public repository) -----------------------
# Raw workbooks are confidential and are NOT part of this repository.
# Place them locally in data/raw/ (git-ignored) or set AQUAVOLTA_DATA_DIR.
import os
from pathlib import Path

DATA_DIR = Path(os.environ.get("AQUAVOLTA_DATA_DIR", "../../data/raw"))
OUTPUT_DIR = Path(os.environ.get("AQUAVOLTA_OUTPUT_DIR", "../../data/processed"))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
res = pd.read_excel(DATA_DIR / "DATA BI.xlsx",sheet_name='ENCAISSEMENTS', header=2)
print(res)

In [ ]:
print(res.shape)

In [ ]:
print(res.info())

In [ ]:
print(res.describe())

In [ ]:
msno.bar(res)

In [ ]:
mode=res[["BANQUE"]].mode().iloc[0]
for coll in mode.index:
    res[coll]=res[coll].fillna(mode[coll])
print(res.info())

In [ ]:
# 2. Kol chay bel numeric (on focus 3la colonnes importantes)
cols_numeric = ["CA €", "tx conv", "CA TND", "paiement TND", "TCL"]
data_num = res[cols_numeric]

# 3. Itérative Imputer
imputer = IterativeImputer(estimator=BayesianRidge(), random_state=0)
data_imputed = imputer.fit_transform(data_num)

# 4. Remplacer les anciennes colonnes bel nouvelles
res[cols_numeric] = data_imputed
print(res.info())

In [ ]:
print(res)

In [ ]:
res["RECU (frais déduits)"] = res["RECU (frais déduits)"].apply(
    lambda x: "PAYE" if isinstance(x, (int, float)) else x)
print(res)

In [ ]:
res["DATE"] = pd.to_datetime(res["DATE"]).dt.strftime("%d/%m/%Y")
print(res)

In [ ]:
print(res["DATE"])

In [ ]:
def clean_date(value):
    try:
       
        if pd.isna(value):
            return pd.NaT

       
        value = str(value).strip()

        parts = value.split("/")
        if len(parts) == 3 and len(parts[2]) < 4:
            
            year = parts[2].zfill(4)  
            if int(year) < 1900:      
                year = "20" + year[-2:]
            value = f"{parts[0]}/{parts[1]}/{year}"

       
        return pd.to_datetime(value, dayfirst=True, errors="coerce")

    except Exception:
        return pd.NaT


res["DATE ENC"] = res["DATE ENC"].apply(clean_date)

res["DATE ENC"] = res["DATE ENC"].dt.strftime("%d/%m/%Y")
print(res)

In [ ]:
res.to_excel(OUTPUT_DIR / "output____ENCAISSEMENT.xlsx", index=False)